This notebook demonstrates how to do basic Merfish spatial analysis using scanpy with data stored in google drive. More information available https://scanpy.readthedocs.io/en/stable/tutorials/spatial/basic-analysis.html and https://squidpy.readthedocs.io/en/stable/notebooks/tutorials/tutorial_merfish.html

In [ ]:
#Mount Google Drive to access your files, if they are stored there.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Set the path where you want to store the files (use your own directory).
import os

In [ ]:
#Replace 'RNAseq_folder' with the path to the folder in your Google Drive or use '/content/' for local storage.
rna_seq_path = '/content/drive/My Drive/RNAseq_folder'
os.chdir(rna_seq_path)

In [ ]:
!pip install scanpy  # Make sure scanpy is installed
!pip install pyarrow.parquet  # Make sure pyarrow.parquet is installed
!pip install scanorama  # Make sure scanorama is installed
!pip install squidpy # Make sure squidpy is installed
!pip install igraph
import scanpy as sc
import squidpy as sq
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

In [ ]:
# 1️⃣ Load main AnnData object (replace by your .h5ad file)
adata = sc.read_h5ad("/content/drive/My Drive/RNAseq_folder/sampleMerfish.h5ad")
print(adata)

# 2️⃣ Load cell metadata
cell_metadata = pd.read_csv("/content/drive/My Drive/RNAseq_folder/cell_metadata.csv", index_col=0)
cell_categories = pd.read_csv("/content/drive/My Drive/RNAseq_folder/cell_categories.csv", index_col=0)
cell_numeric_categories = pd.read_csv("/content/drive/My Drive/RNAseq_folder/cell_numeric_categories.csv", index_col=0)

# 3️⃣ Join metadata to adata.obs (only new columns to avoid overwriting)
for df in [cell_metadata, cell_categories, cell_numeric_categories]:
    new_cols = [c for c in df.columns if c not in adata.obs.columns]
    adata.obs = adata.obs.join(df[new_cols], how="left")

print(adata.obs.head())

# 4️⃣ Load raw cell boundaries from cell_metadata
boundary_cols = [c for c in cell_metadata.columns if "Cellbound" in c and "raw" in c]
cell_boundaries_raw = cell_metadata[boundary_cols]

# Make sure index matches adata.obs_names
cell_boundaries_raw.index = cell_metadata.index.astype(str)
cell_boundaries_raw = cell_boundaries_raw.reindex(adata.obs_names)

# Store in AnnData
adata.obsm["cell_boundaries"] = cell_boundaries_raw
print(adata.obsm["cell_boundaries"].head())

# 5️⃣ Add spatial coordinates (required by Squidpy)
if "center_x" in adata.obs.columns and "center_y" in adata.obs.columns:
    adata.obsm["spatial"] = adata.obs[["center_x", "center_y"]].values
    print("Spatial coordinates added to adata.obsm['spatial']")
else:
    print("Warning: 'center_x' and/or 'center_y' not found in adata.obs")

In [ ]:
#duplicate the previous cell if multiple samples (e.g. adata1, adata2...) and concatenate them using adata = sc.concat([adata1,adata2])

In [ ]:
sc.pp.normalize_per_cell(adata, counts_per_cell_after=1e6)
sc.pp.log1p(adata)
sc.pp.pca(adata, n_comps=15)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.tl.leiden(
    adata,
    key_added="clusters",
    resolution=0.5,
    n_iterations=2,
    flavor="igraph",
    directed=False,
)

In [ ]:
adata

In [ ]:
sc.pl.umap(adata, color="clusters")
sc.pl.embedding(adata, basis="spatial", color="clusters")

In [ ]:
sc.pl.spatial(adata, img_key="hires", color=["clusters", "CD8A"], spot_size=25) #replace CD8A with your genes of interest

In [ ]:
sc.tl.rank_genes_groups(adata, "clusters", method="t-test")
sc.pl.rank_genes_groups_heatmap(adata, groups="0", n_genes=10, groupby="clusters") #replace 0 by your cluster of interest

In [ ]:
sq.gr.spatial_neighbors(adata, coord_type="generic", spatial_key="spatial")
sq.gr.nhood_enrichment(adata, cluster_key="clusters")
sq.pl.nhood_enrichment(
    adata, cluster_key="clusters", method="single", cmap="inferno", vmin=-50, vmax=100
)

In [ ]:
sc.pl.embedding(
    adata,
    basis="spatial",
    groups=["0", "1"], #replace 0 and 1 by your clusters of interest
    na_color=(1, 1, 1, 0),
    projection="2d",
    color="clusters",
)

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="clusters")
sc.pl.rank_genes_groups(adata, groupby="clusters")

In [ ]:
sq.gr.spatial_autocorr(adata, mode="moran")
adata.uns["moranI"].head()
sq.pl.spatial_scatter(
    adata, shape=None, color=["SPP1", "CD3E"], size=3 #replace CD3E and SPP1 with your genes of interest
)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from sklearn.decomposition import PCA

# --- Original gene list (immune clusters) + HLA genes ---
gene_list1 = [
    "PDK4", "CCL26", "CX3CL1", "PGLYRP1", "CD4", "SNAI2", "TNFRSF17", "ICAM3", "TBX21", "FAP",
    "NFKB2", "LAG3", "TGFBR3", "MMP11", "XBP1", "IL2RB", "CTSG", "GZMH", "GZMB", "NFKBIA",
    "MMP2", "CCL22", "TSC2", "TGFB1", "CD79A", "NKG7", "HGF", "SERPINE1", "ACTA2", "MPO",
    "CSF3", "WNT3", "CCL2", "CCL1", "COL1A1", "VTN", "NFKB1", "CCND1", "IL23A", "CDKN1B",
    "IFNG", "KLRB1", "LOX", "GZMK", "IL12B", "SELL", "TGFB3", "DUSP1", "EGR1", "KLRK1",
    "TNFSF10", "CXCR4", "TWIST1", "LRP1", "SNAI1", "CEACAM8", "CDKN1A", "SOX9", "TNFSF9", "CD70",
    "FFAR2", "CCR7", "KLF2", "SMO", "RAF1", "PLVAP", "BST2", "LGR6", "CCNB1", "PDGFRA",
    "APC", "TBX3", "CDK4", "IL6", "BLK", "FGFBP2", "TLR2", "MMP7", "LYZ", "LRP6",
    "VWF", "PROX1", "CDH1", "PDGFRB", "FLT4", "CREBBP", "IL1B", "BRD4", "ELANE", "EPCAM",
    "AKT3", "GNLY", "KDR", "PIK3CA", "LAMC2", "MCM6", "FGF2", "STAT3", "ICAM1", "PPARGC1A",
    "CXCL9", "MCM2", "MSH3", "EGF", "STAP1", "LAMP3", "CDK6", "PMS2", "KLRG1", "LGR5",
    "CDK2", "RB1", "ITGAX", "TP53", "IFNAR1", "FCGR2A", "ACKR3", "FABP2", "SFRP2", "GZMA",
    "EGFR", "DUSP6", "TNFSF4", "YAP1", "ITGA1", "WNT3A", "FZD7", "CSF1R", "KIT", "PREX2",
    "SLC13A3", "TNFRSF13C", "CXCR5", "CCR5", "MAML1", "ITGA5", "CXCL16", "LRP5", "CXCR1", "CXCL5",
    "CCR1", "CSF2", "PLK1", "STAT6", "CD3D", "CD14", "MZB1", "ITGB1", "JUNB", "CEBPB",
    "FOS", "CXCR6", "IRS1", "ESCO2", "CCL11", "BMP1", "TMEM37", "CXCL10", "CXCL11", "IGF1",
    "CXCL8", "BCL2L1", "HAVCR2", "CTSW", "TLR1", "XCR1", "VEGFB", "HRAS", "CD248", "KRAS",
    "FOSL1", "NOS2", "MMP1", "CD209", "NCAM1", "ZBED2", "DERL3", "RORC", "CD276", "CXCR2",
    "ZEB1", "HIF1A", "ANGPT2", "SOX2", "CCR8", "KIR3DL2", "MARCO", "FBLN1", "IFITM1", "CSF1",
    "EPHB3", "CD86", "STING1", "SOCS3", "CCR4", "IL3RA", "PDGFB", "ASCL2", "TBK1", "PTGDR2",
    "IL1R2", "BCL2", "PDCD1", "PKM", "TBX10", "CA7", "SLC26A3", "TNC", "CAV1", "ATF3",
    "INSR", "CCR6", "CLEC14A", "JAK1", "HLA-DQA1", "E2F1", "IDH1", "BAX", "GATA3", "DDIT3",
    "CDCA7", "CD5", "KLRC1", "PTPRC", "CTNNB1", "BIRC5", "ATR", "EZH2", "CD8A", "BRCA1",
    "PROK2", "COL11A1", "TAP1", "PKIB", "TP63", "LAMB3", "IRF5", "CCNE1", "NF1", "EPHA2",
    "TGFBR2", "KIR2DL4", "FGF1", "MFAP5", "FOXM1", "HLA-DRB1", "FCER2", "CLEC4C", "TLR9", "STAT1",
    "CCL28", "TGM2", "MUC2", "SOD2", "CR2", "PTGS2", "FASLG", "IFNGR1", "XCL1", "SH2D1B",
    "ADAMTS4", "CD244", "ARG1", "FCER1A", "CD1E", "CD1B", "FCRL5", "LMNA", "IL6R", "MKI67",
    "S100A9", "CD2", "NRAS", "VCAM1", "CD40LG", "JUN", "AURKA", "COL5A1", "PTEN", "MMRN2",
    "VEGFA", "CD40", "MMP9", "TREM2", "MAFB", "HDAC1", "CXCR3", "DES", "DKK1", "BAK1",
    "CD28", "C1QC", "HLA-DMA", "NRP1", "TAP2", "TGFBR1", "PLA2G2A", "COL4A1", "NCR3", "FOXP3",
    "HLA-C", "BMI1", "MTOR", "THBD", "SPRY2", "MYC", "CA9", "KITLG", "CYBB", "PCNA",
    "CD83", "TNFRSF4", "TNFRSF18", "LDHA", "IL2RA", "TEK", "IFNB1", "ELN", "DNMT3A", "IRF4",
    "CD274", "IFNGR2", "MSR1", "IFNAR2", "TIGIT", "GPX3", "CD8B", "IL4I1", "AKT2", "STAT4",
    "CD79B", "POU2AF1", "CCL8", "CCL7", "CLCA1", "MMRN1", "SPP1", "HLA-DRA", "LGALS9", "CXCL1",
    "CXCL12", "MYBL2", "ITGA4", "NFE2L2", "TEAD4", "PDCD1LG2", "MET", "ITGB2", "SHARPIN", "CX3CR1",
    "CCR2", "PDGFA", "ROS1", "CEACAM1", "TSC1", "CHEK2", "RELA", "ICOSLG", "AKT1", "CD207",
    "PDK1", "FGFR3", "HLA-B", "CMKLR1", "PRKCA", "IDO2", "PPARD", "HLA-DPA1", "SIGLEC1", "AMOTL2",
    "FGFR1", "TRAT1", "SELP", "CTLA4", "IKZF4", "TAPBP", "MLH1", "ICOS", "NEDD4", "FCGR3A",
    "CHEK1", "CD1C", "IL5RA", "TNF", "EOMES", "IKZF2", "FN1", "PLOD2", "ZAP70", "CSF3R",
    "DIABLO", "IL12RB2", "WWTR1", "SRPRB", "IL12A", "IL13", "HDAC3", "EPHA4", "ARAF", "VSIR",
    "TMEM59", "COL6A3", "WNT5A", "BTLA", "SRC", "NOS3", "BUB1", "CD80", "MUC4", "TGFB2",
    "ENG", "MUC1", "CASP8", "PIK3CG", "BRAF", "HLA-DPB1", "CSF2RA", "PDGFC", "LEF1", "TGFBI",
    "CXCL2", "RGMB", "ANGPT1", "IDO1", "ITK", "IKBKB", "PAX5", "PTK2", "TEAD1", "FLI1",
    "CD3E", "MS4A1", "CD3G", "TCF7L2", "SMOC2", "ETS1", "CD22", "IL22", "CD19", "MSH6",
    "CD27", "ERBB2", "CD163", "NLRC5", "MSH2", "PRTN3", "ITGAM", "CCR3", "ROR1", "ERBB3",
    "SELPLG", "PGF", "NDUFA4L2", "TCL1A", "IGF1R", "PECAM1", "MAP2K1", "MRC1", "MMP12", "MYH11",
    "ICAM2", "CD226", "TNFRSF13B", "CD68", "AURKB", "SMAD2", "SMARCA4", "STAT5A", "CCR10", "DNMT1",
    "NCR1", "IRF3", "CCL5", "IL21", "TRAC", "CCL3", "CCL4", "CDH5", "TNFRSF9", "RET",
    "FLT1", "KLRF1", "CD160", "EPHB4", "CCL17", "CIITA", "CLDN5", "CD177", "VEGFC", "AXIN2",
    "PDPN", "IL4", "TRBC1", "SERPINA1", "FGFR2", "ATM", "PRF1", "IL10", "IL17A", "NOTCH1"

]

# --- Add all HLA genes ---
hla_genes = [
    'HLA-A','HLA-B','HLA-C','HLA-E','HLA-F','HLA-G',
    'HLA-DRA','HLA-DRB1','HLA-DPA1','HLA-DPB1'
]
gene_list1 += hla_genes

# --- Replace "-" with "_" to match var_names ---
gene_list1_mod = [g.replace("-", "_") for g in gene_list1]

# --- Filter genes present in adata ---
gene_list = [g for g in gene_list1_mod if g in adata.var_names]
print(f"Using {len(gene_list)} genes out of {len(gene_list1)}")
print("Genes included:", gene_list)

# --- Extract expression matrix ---
X = adata[:, gene_list].X
if hasattr(X, "toarray"):  # handle sparse
    X = X.toarray()

# --- Normalize each gene 0–1 ---
X = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0) + 1e-9)

# --- Spatial coordinates ---
coords = adata.obsm["spatial"]

# ============================================================
# 1️⃣ Pairwise correlations and p-values
# ============================================================
n = len(gene_list)
corr_matrix = np.zeros((n, n))
pval_matrix = np.ones((n, n))

for i in range(n):
    for j in range(n):
        r, p = pearsonr(X[:, i], X[:, j])
        corr_matrix[i, j] = r
        pval_matrix[i, j] = p

corr_df = pd.DataFrame(corr_matrix, index=gene_list, columns=gene_list)
pval_df = pd.DataFrame(pval_matrix, index=gene_list, columns=gene_list)

# ============================================================
# 2️⃣ Significant correlations
# ============================================================
sig_mask = (pval_df < 0.05) & (abs(corr_df) > 0.1)
significant_genes = sorted(
    set([gene for i, g1 in enumerate(gene_list)
         for j, g2 in enumerate(gene_list)
         if sig_mask.iloc[i, j] and i != j
         for gene in [g1, g2]])
)

print(f"✅ {len(significant_genes)} genes with significant colocalization:")
print(significant_genes)

if len(significant_genes) < 2:
    raise ValueError("No significant gene pairs found — try lowering the r or p-value threshold.")

# Subset correlation matrix
corr_sig = corr_df.loc[significant_genes, significant_genes]
sig_mask_sig = sig_mask.loc[significant_genes, significant_genes]

# ============================================================
# 3️⃣ Heatmap with stars
# ============================================================
plt.figure(figsize=(10, 8))
annot_matrix = np.where(sig_mask_sig, "*", "")
sns.heatmap(
    corr_sig,
    cmap="vlag",
    center=0,
    square=True,
    annot=annot_matrix,
    fmt='',
    linewidths=0.3,
    cbar_kws={"label": "Pearson r"}
)
plt.xticks(rotation=90, ha="center", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.title("Significant spatial gene colocalization (|r|>0.1, p<0.05)", fontsize=14, pad=20)
plt.tight_layout()
plt.show()

# ============================================================
# 4️⃣ Clustermap of significant genes with stars
# ============================================================
# Create annotation matrix: "*" if significant, "" otherwise
annot_matrix = np.where(sig_mask_sig, "*", "")

g = sns.clustermap(
    corr_sig,
    cmap="vlag",
    center=0,
    linewidths=0.3,
    figsize=(12, 12),
    xticklabels=True,
    yticklabels=True,
    annot=annot_matrix,      # <-- add annotations
    fmt='',                  # <-- no formatting for text
)

# Change font size of tick labels
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), fontsize=8)
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=8)

plt.suptitle("Clusters of significantly co-localized genes (|r|>0.1, p<0.05)", y=1.02, fontsize=14)
plt.show()